# 04. 피처 엔지니어링 — fastText 기반 seed 확장

03에서 전처리한 `dart_preprocessed.csv`(`회사이름`, `esg_year`, `raw_text`, `clean_text`, `kiwi_text`)와 `data/raw/seed_dictionary.csv`를 사용해, **30개 seed 단어를 fastText 유사어로 확장**한 expanded dictionary 초안을 만듭니다.

**핵심 흐름**

1. `kiwi_text`를 fastText 학습 문장으로 변환 (긴 문서는 180토큰 chunk)
2. fastText 모델 학습 (skip-gram, subword n-gram)
3. seed 30개 → fastText query term 매핑
4. seed별 cosine similarity top-N 후보 추출 (저장 임계값 0.40)
5. **임계치 θ ∈ {0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75} 별 후보 규모와 잡음 비율 비교**
6. θ=0.60 기준 expanded dictionary 검토 템플릿 생성

**fastText 파라미터** (기존 05 노트북 그대로 차용 — Bao et al. 2024 권장값)

- `vector_size=150`, `window=6`, `min_count=2`, `sg=1` (skip-gram)
- `negative=10`, `sample=1e-4`, `min_n=2`, `max_n=5`, `epochs=12`, `seed=42`

**산출 CSV** (임계치 확인용 중심)

- `fasttext_threshold_summary.csv` — θ별 후보 수, 잡음 비율, dimension 분포
- `fasttext_seed_candidates.csv` — seed×candidate 전체 후보 (similarity ≥ 0.40)
- `expanded_dictionary_review_template.csv` — θ=0.60 검토 템플릿


## 1. 환경 설정과 입력 파일 확인


In [1]:
import os
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from gensim.models import FastText
from kiwipiepy import Kiwi

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
PREPROCESSED_CSV = DATA_PROCESSED / 'dart_preprocessed.csv'
SEED_DICTIONARY_CSV = DATA_RAW / 'seed_dictionary.csv'
STOPWORDS_PATH = DATA_RAW / 'stopwords_ko_esg.txt'

OUTPUT_DIR = DATA_PROCESSED / 'feature_engineering'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

THRESHOLD_SUMMARY_CSV = OUTPUT_DIR / 'fasttext_threshold_summary.csv'
SEED_CANDIDATES_CSV = OUTPUT_DIR / 'fasttext_seed_candidates.csv'
REVIEW_TEMPLATE_CSV = OUTPUT_DIR / 'expanded_dictionary_review_template.csv'
FASTTEXT_MODEL_PATH = OUTPUT_DIR / 'fasttext_seed_expansion.model'

for path in [PREPROCESSED_CSV, SEED_DICTIONARY_CSV, STOPWORDS_PATH]:
    assert path.exists(), f'필수 입력 누락: {path}'

print('입력:', PREPROCESSED_CSV)
print('seed:', SEED_DICTIONARY_CSV)
print('출력 폴더:', OUTPUT_DIR)


입력: /Users/home1/esg-dart-project/data/processed/dart_preprocessed.csv
seed: /Users/home1/esg-dart-project/data/raw/seed_dictionary.csv
출력 폴더: /Users/home1/esg-dart-project/data/processed/feature_engineering


## 2. 데이터 로딩

03에서 만든 `kiwi_text`는 이미 불용어 제거·로마숫자 제거·NFKC 정규화·소문자화가 적용된 공백 구분 토큰 문자열입니다. 추가 가공 없이 fastText 입력으로 쓸 수 있습니다.


In [2]:
df = pd.read_csv(PREPROCESSED_CSV)
seed_dictionary = pd.read_csv(SEED_DICTIONARY_CSV)

with STOPWORDS_PATH.open('r', encoding='utf-8') as f:
    stopwords = {
        unicodedata.normalize('NFKC', line.strip()).lower()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }
ROMAN_STOPWORDS = {'i', 'ii', 'iii', 'iv', 'v', 'vi', 'vii', 'viii', 'ix', 'x', 'xi', 'xii'}
stopwords |= ROMAN_STOPWORDS

df['kiwi_text'] = df['kiwi_text'].fillna('').astype(str)
n_valid = (df['kiwi_text'].str.strip() != '').sum()

print(f'전처리 입력 행 수: {len(df)}')
print(f'kiwi_text 비어있지 않은 행: {n_valid}')
print(f'seed: {len(seed_dictionary)}개')
print(f'불용어: {len(stopwords)}개')
df[['회사이름', 'esg_year']].head()


전처리 입력 행 수: 381
kiwi_text 비어있지 않은 행: 381
seed: 30개
불용어: 45개


,회사이름,esg_year
0,동화약품,2023
1,동화약품,2024
2,동화약품,2025
3,KR모터스,2023
4,KR모터스,2024


## 3. fastText 학습용 문장 만들기

긴 보고서를 통째로 한 "문장"으로 넣으면 로컬 문맥이 흐려집니다. 180토큰 chunk로 잘라 각 chunk를 한 문장으로 취급합니다 (window=6 안에서 의미 학습).


In [3]:
CHUNK_SIZE = 180
MIN_DOC_TOKENS = 20
TOKEN_RE = re.compile(r'^[가-힣A-Za-z]+$')


def text_to_tokens(text: str) -> list[str]:
    if not isinstance(text, str):
        return []
    return [t for t in text.split() if TOKEN_RE.fullmatch(t) and t not in stopwords and len(t) > 1]


def chunk_tokens(tokens: list[str], size: int = CHUNK_SIZE) -> list[list[str]]:
    return [tokens[i:i + size] for i in range(0, len(tokens), size) if len(tokens[i:i + size]) >= MIN_DOC_TOKENS]


doc_tokens = df['kiwi_text'].map(text_to_tokens)
sentences: list[list[str]] = []
for tokens in doc_tokens:
    sentences.extend(chunk_tokens(tokens))

total_tokens = sum(len(t) for t in doc_tokens)
print(f'문서 수 (≥{MIN_DOC_TOKENS}토큰): {sum(1 for t in doc_tokens if len(t) >= MIN_DOC_TOKENS)}')
print(f'학습 chunk 수: {len(sentences)}')
print(f'총 토큰: {total_tokens:,}')
print(f'chunk 평균 길이: {np.mean([len(s) for s in sentences]):.1f}')


문서 수 (≥20토큰): 381
학습 chunk 수: 29990
총 토큰: 5,371,093
chunk 평균 길이: 179.1


## 4. fastText 모델 학습

skip-gram fastText는 드문 ESG 용어와 한국어 복합어에 유리합니다. `min_count=2`, 문자 n-gram `2~5`로 OOV 보완.


In [4]:
MODEL_PARAMS = {
    'vector_size': 150,
    'window': 6,
    'min_count': 2,
    'workers': max(1, min(8, (os.cpu_count() or 2) - 1)),
    'sg': 1,
    'negative': 10,
    'sample': 1e-4,
    'min_n': 2,
    'max_n': 5,
    'seed': 42,
    'epochs': 12,
}

ft_model = FastText(sentences=sentences, **MODEL_PARAMS)

print(f'fastText vocabulary: {len(ft_model.wv.key_to_index):,}개')
print('상위 30개 토큰:', list(ft_model.wv.key_to_index)[:30])


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

fastText vocabulary: 43,200개
상위 30개 토큰: ['이사', '금융', '위원회', '임원', '단위', '개발', '감사', '기타', '위하', '백만', '위험', '투자', '경영', '주요', '지급', '사외', '주주', '당기', '기업', '통하', '주식', '상품', '거래', '산업', '해당', '이사회', '승인', '상근', '관하', '등기']


## 5. seed → fastText query term

`seed_term`(`재생에너지`)과 `pattern`(`net zero|net-zero`)을 분해해 Kiwi로 토큰화한 뒤, fastText가 벡터를 만들 수 있는 표현만 query로 채택합니다.


In [5]:
kiwi = Kiwi()
KEEP_POS_PREFIXES = ('N', 'V')
KEEP_POS_TAGS = {'SL'}


def normalize_word(word: object) -> str:
    return unicodedata.normalize('NFKC', str(word)).lower().strip()


def tokenize_seed_phrase(phrase: object) -> list[str]:
    if not isinstance(phrase, str) or not phrase.strip():
        return []
    phrase = normalize_word(phrase)
    out: set[str] = set()
    compact = re.sub(r'[^가-힣A-Za-z]+', '', phrase)
    if len(compact) > 1:
        out.add(compact)
    for part in re.split(r'[^가-힣A-Za-z]+', phrase):
        if len(part) > 1 and TOKEN_RE.fullmatch(part):
            out.add(part)
    for token in kiwi.tokenize(phrase):
        word = normalize_word(token.form)
        if len(word) <= 1 or not TOKEN_RE.fullmatch(word):
            continue
        if not (token.tag.startswith(KEEP_POS_PREFIXES) or token.tag in KEEP_POS_TAGS):
            continue
        out.add(word)
    return sorted(out)


def can_query(term: str) -> bool:
    try:
        _ = ft_model.wv[term]
        return True
    except KeyError:
        return False


def seed_query_terms(row: pd.Series) -> list[str]:
    phrases = [row['seed_term']]
    if isinstance(row.get('pattern'), str):
        phrases.extend(row['pattern'].split('|'))
    terms: list[str] = []
    for phrase in phrases:
        terms.extend(tokenize_seed_phrase(phrase))
    return sorted({t for t in terms if t not in stopwords and can_query(t)})


seed_dictionary['fasttext_query_terms'] = seed_dictionary.apply(seed_query_terms, axis=1)
seed_dictionary['fasttext_query_count'] = seed_dictionary['fasttext_query_terms'].map(len)

print('seed별 query 수:')
print(seed_dictionary[['dimension', 'seed_term', 'fasttext_query_count']].to_string(index=False))

missing = seed_dictionary[seed_dictionary['fasttext_query_count'] == 0]
if not missing.empty:
    print('\n⚠ query 생성 실패 seed:')
    print(missing[['dimension', 'seed_term', 'pattern']].to_string(index=False))


seed별 query 수:
dimension seed_term  fasttext_query_count
        E        탄소                     1
        E      온실가스                     2
        E      탄소중립                     3
        E       넷제로                     4
        E     재생에너지                     6
        E       에너지                     1
        E        전력                     3
        E       폐기물                     1
        E       재활용                     5
        E        폐수                     3
        S        안전                     1
        S      산업재해                     4
        S      중대재해                     3
        S       임직원                     3
        S        노동                     2
        S        인권                     1
        S      교육훈련                     3
        S       협력사                     1
        S       공급망                     4
        S      지역사회                     4
        G       이사회                     2
        G      사외이사                     3
        G     감사위원회

## 6. seed별 유사어 후보 추출

각 query term에 대해 topN=200 후보를 가져와 cosine similarity ≥ 0.40인 것만 저장합니다. 후속 임계치 비교 단계에서 더 높은 θ로 자르면 됩니다.

**필터링**
- seed 자기 자신, 불용어, 한 글자, 한글+영문 혼합 제외
- 한글/영문 단독 토큰만 보존


In [6]:
TOPN_PER_QUERY_TERM = 200
MIN_SIM_TO_SAVE = 0.40

ALL_SEED_TERMS: set[str] = set()
for row in seed_dictionary.itertuples(index=False):
    ALL_SEED_TERMS.update(tokenize_seed_phrase(row.seed_term))
    if isinstance(row.pattern, str):
        for phrase in row.pattern.split('|'):
            ALL_SEED_TERMS.update(tokenize_seed_phrase(phrase))


def candidates_for_seed(query_terms: list[str]) -> list[dict]:
    if not query_terms:
        return []
    best: dict[str, dict] = {}
    for q in query_terms:
        try:
            similar = ft_model.wv.most_similar(q, topn=TOPN_PER_QUERY_TERM)
        except KeyError:
            continue
        for word, sim in similar:
            word = normalize_word(word)
            sim = float(sim)
            if sim < MIN_SIM_TO_SAVE:
                continue
            if word in stopwords or word in ALL_SEED_TERMS or len(word) <= 1:
                continue
            if not TOKEN_RE.fullmatch(word):
                continue
            # 한글+영문 혼합 제외
            if re.search(r'[가-힣]', word) and re.search(r'[A-Za-z]', word):
                continue
            if word not in best or sim > best[word]['similarity']:
                best[word] = {'candidate': word, 'similarity': sim, 'source_query_term': q}
    return sorted(best.values(), key=lambda r: r['similarity'], reverse=True)


rows = []
for seed_row in seed_dictionary.itertuples(index=False):
    for rank, cand in enumerate(candidates_for_seed(seed_row.fasttext_query_terms), start=1):
        rows.append({
            'dimension': seed_row.dimension,
            'seed_term': seed_row.seed_term,
            'rank_within_seed': rank,
            'candidate': cand['candidate'],
            'similarity': cand['similarity'],
            'source_query_term': cand['source_query_term'],
        })

seed_candidates = pd.DataFrame(rows)
print(f'seed-candidate pair 수: {len(seed_candidates):,}')
print(f'고유 후보 단어 수: {seed_candidates["candidate"].nunique():,}')
print('\ndimension별 후보:')
print(seed_candidates.groupby('dimension').agg(
    pairs=('candidate', 'size'),
    unique_candidates=('candidate', 'nunique'),
    mean_sim=('similarity', 'mean'),
    max_sim=('similarity', 'max'),
).round(3).to_string())


seed-candidate pair 수: 13,137
고유 후보 단어 수: 7,286

dimension별 후보:
           pairs  unique_candidates  mean_sim  max_sim
dimension                                             
E           4215               2654     0.574    0.957
G           4685               3002     0.633    0.982
S           4237               3124     0.555    0.869


## 7. **임계치 θ별 후보 규모와 잡음 비율 비교**

θ를 낮추면 후보가 늘고 잡음도 늘어납니다. 7개 θ 값에서 unique candidate 수와 상위 100개 기준 잡음 추정 비율을 표로 정리합니다.

**잡음 휴리스틱** — `heuristic_noise_flag`
- 12자 초과 한글 복합어
- ESG 키워드가 없는 짧은 영문
- 다수 seed에 걸쳐 있지만 최대 유사도 < 0.65 ("광범위 저특이성")


In [7]:
ESG_EN_HINTS = {
    'carbon', 'climate', 'netzero', 'renewable', 'energy', 'recycle', 'recycling',
    'waste', 'water', 'safety', 'human', 'rights', 'supply', 'compliance',
    'ethics', 'board', 'audit', 'shareholder', 'governance', 'emission', 'green',
}
CORE_KO_HINTS = {
    '탄소', '온실', '기후', '재생', '에너지', '전력', '폐기', '재활용', '폐수', '배출',
    '안전', '재해', '임직원', '직원', '근로자', '노동', '인권', '교육', '훈련', '협력', '공급망',
    '이사회', '사외', '이사', '감사', '위원회', '독립', '윤리', '준법', '부패', '주주', '의결권',
}


def noise_reason(word: str, seed_support: int, dim_support: int, max_sim: float) -> str:
    reasons = []
    if re.fullmatch(r'[가-힣]+', word) and len(word) > 12 and not any(h in word for h in CORE_KO_HINTS):
        reasons.append('long_compound')
    if re.fullmatch(r'[a-z]+', word) and len(word) <= 3 and word not in {'esg', 'ghg', 'iso', 'co2'}:
        reasons.append('short_english')
    if re.fullmatch(r'[a-z]+', word) and len(word) > 3 and not any(h in word for h in ESG_EN_HINTS):
        reasons.append('english_no_esg_hint')
    if seed_support >= 8 and dim_support >= 2 and max_sim < 0.65:
        reasons.append('broad_low_specificity')
    return '|'.join(reasons)


def aggregate_at_theta(theta: float) -> pd.DataFrame:
    sub = seed_candidates.loc[seed_candidates['similarity'] >= theta]
    if sub.empty:
        return pd.DataFrame(columns=['candidate', 'primary_dimension', 'max_similarity',
                                       'seed_support_count', 'dimension_support_count',
                                       'heuristic_noise_reason', 'heuristic_noise_flag'])
    out = []
    for candidate, group in sub.groupby('candidate'):
        dim_scores = group.groupby('dimension')['similarity'].max().sort_values(ascending=False)
        max_sim = float(group['similarity'].max())
        seed_support = int(group[['dimension', 'seed_term']].drop_duplicates().shape[0])
        dim_support = int(group['dimension'].nunique())
        reason = noise_reason(candidate, seed_support, dim_support, max_sim)
        out.append({
            'candidate': candidate,
            'primary_dimension': dim_scores.index[0],
            'max_similarity': max_sim,
            'mean_similarity': float(group['similarity'].mean()),
            'seed_support_count': seed_support,
            'dimension_support_count': dim_support,
            'supporting_seeds': '|'.join(sorted(group['seed_term'].unique())),
            'heuristic_noise_reason': reason,
            'heuristic_noise_flag': bool(reason),
        })
    return pd.DataFrame(out).sort_values('max_similarity', ascending=False).reset_index(drop=True)


THRESHOLDS = [0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75]
TOP_REVIEW = 100

summary_rows = []
aggregated_by_theta: dict[float, pd.DataFrame] = {}
for theta in THRESHOLDS:
    agg = aggregate_at_theta(theta)
    aggregated_by_theta[theta] = agg
    top = agg.head(TOP_REVIEW)
    summary_rows.append({
        'theta': theta,
        'unique_candidates': len(agg),
        'top100_noise_count': int(top['heuristic_noise_flag'].sum()) if not top.empty else 0,
        'top100_noise_ratio': round(float(top['heuristic_noise_flag'].mean()), 3) if not top.empty else np.nan,
        'E_terms': int((agg['primary_dimension'] == 'E').sum()),
        'S_terms': int((agg['primary_dimension'] == 'S').sum()),
        'G_terms': int((agg['primary_dimension'] == 'G').sum()),
    })

threshold_summary = pd.DataFrame(summary_rows)
print('임계치 θ별 후보 규모와 잡음 비율:')
print(threshold_summary.to_string(index=False))


임계치 θ별 후보 규모와 잡음 비율:
 theta  unique_candidates  top100_noise_count  top100_noise_ratio  E_terms  S_terms  G_terms
  0.45               7108                  40                 0.4     2196     2202     2710
  0.50               6455                  40                 0.4     1949     1893     2613
  0.55               5267                  40                 0.4     1537     1389     2341
  0.60               3502                  40                 0.4      875      771     1856
  0.65               2256                  40                 0.4      511      265     1480
  0.70               1190                  40                 0.4      153       77      960
  0.75                578                  40                 0.4       81       32      465


## 8. θ=0.60 expanded dictionary 검토 템플릿

θ=0.60은 recall과 precision 사이의 시작점입니다. E/S/G 각 100개씩 상위 후보를 모아 사람이 `keep/drop/revise` 결정을 채울 수 있는 검토 템플릿을 만듭니다.


In [8]:
FINAL_THETA = 0.60
TOP_PER_DIMENSION = 100

base = aggregated_by_theta[FINAL_THETA]
review = pd.concat([
    base.loc[base['primary_dimension'] == dim].head(TOP_PER_DIMENSION)
    for dim in ['E', 'S', 'G']
], ignore_index=True)

review.insert(0, 'term_type', 'fasttext_candidate')
review.insert(1, 'review_decision', '')
review.insert(2, 'review_reason', '')
review['theta'] = FINAL_THETA

# seed 30개는 keep 상태로 위에 붙입니다.
seed_rows = seed_dictionary[['dimension', 'seed_term']].rename(
    columns={'dimension': 'primary_dimension', 'seed_term': 'candidate'}
)
seed_rows.insert(0, 'term_type', 'seed')
seed_rows.insert(1, 'review_decision', 'keep')
seed_rows.insert(2, 'review_reason', 'provided_seed')
for col in review.columns:
    if col not in seed_rows.columns:
        seed_rows[col] = ''
seed_rows = seed_rows[review.columns]

review_template = pd.concat([seed_rows, review], ignore_index=True)

print(f'θ={FINAL_THETA} 기준 unique 후보: {len(base):,}')
print(f'E/S/G 각 {TOP_PER_DIMENSION}개 + seed 30개 → 템플릿 행 수: {len(review_template):,}')
print('\n검토 템플릿 상위 5행:')
print(review_template.head().to_string(index=False))


θ=0.6 기준 unique 후보: 3,502
E/S/G 각 100개 + seed 30개 → 템플릿 행 수: 330

검토 템플릿 상위 5행:
term_type review_decision review_reason candidate primary_dimension max_similarity mean_similarity seed_support_count dimension_support_count supporting_seeds heuristic_noise_reason heuristic_noise_flag theta
     seed            keep provided_seed        탄소                 E                                                                                                                                             
     seed            keep provided_seed      온실가스                 E                                                                                                                                             
     seed            keep provided_seed      탄소중립                 E                                                                                                                                             
     seed            keep provided_seed       넷제로                 E                 

## 9. 산출물 저장

임계치 확인용 핵심 CSV 3개와 fastText 모델을 저장합니다.


In [9]:
threshold_summary.to_csv(THRESHOLD_SUMMARY_CSV, index=False, encoding='utf-8-sig')
seed_candidates.to_csv(SEED_CANDIDATES_CSV, index=False, encoding='utf-8-sig')
review_template.to_csv(REVIEW_TEMPLATE_CSV, index=False, encoding='utf-8-sig')
ft_model.save(str(FASTTEXT_MODEL_PATH))

print('저장 완료:')
print(f'  {THRESHOLD_SUMMARY_CSV}  ({len(threshold_summary)} rows)')
print(f'  {SEED_CANDIDATES_CSV}  ({len(seed_candidates):,} rows)')
print(f'  {REVIEW_TEMPLATE_CSV}  ({len(review_template)} rows)')
print(f'  {FASTTEXT_MODEL_PATH}')


저장 완료:
  /Users/home1/esg-dart-project/data/processed/feature_engineering/fasttext_threshold_summary.csv  (7 rows)
  /Users/home1/esg-dart-project/data/processed/feature_engineering/fasttext_seed_candidates.csv  (13,137 rows)
  /Users/home1/esg-dart-project/data/processed/feature_engineering/expanded_dictionary_review_template.csv  (330 rows)
  /Users/home1/esg-dart-project/data/processed/feature_engineering/fasttext_seed_expansion.model


## 다음 단계

`expanded_dictionary_review_template.csv`를 열어 `review_decision`(keep/drop/revise)과 `review_reason`을 사람이 채운 뒤, 확정된 expanded dictionary를 사용해 TF-IDF 합산 점수를 만들면 됩니다.

임계치 선택 근거는 `fasttext_threshold_summary.csv`의 `top100_noise_ratio`와 dimension 분포 균형을 보고 정합니다. 일반적으로 θ가 낮으면 후보가 많지만 잡음이 늘고, 높으면 후보가 적지만 ESG 핵심어를 놓칠 위험이 있습니다.
